# 03 — Federated Evaluation & Analysis

This notebook analyses the federated training results:
- Convergence behaviour
- Per-client performance
- Communication cost
- Privacy properties

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.config import N_CLIENTS, NUM_ROUNDS, MODELS_DIR, PLOTS_DIR, FEATURE_COLUMNS
from src.data_utils import load_client_data, prepare_client_tensors, get_global_scaler
from src.models import get_model_parameters
from src.metrics import (
    evaluate_model,
    compute_convergence_round,
    compute_communication_cost,
)
from src.privacy import estimate_privacy_budget_spent

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Training History

In [ ]:
with open(MODELS_DIR / 'fl_metrics_history.json') as f:
    data = json.load(f)

history = data['fl_metrics_history']
metrics_df = pd.DataFrame(history)
print(f'Loaded {len(history)} rounds of metrics')
print(metrics_df[['round','loss','accuracy','f1','roc_auc']].tail())

## 2. Convergence Analysis

In [ ]:
conv_round = compute_convergence_round(history, metric='accuracy', threshold_fraction=0.90)
final_acc = metrics_df['accuracy'].iloc[-1]
final_auc = metrics_df['roc_auc'].iloc[-1]

print(f'Final accuracy:   {final_acc:.4f}')
print(f'Final ROC-AUC:    {final_auc:.4f}')
print(f'Convergence round (90% of final): {conv_round}')
print(f'Accuracy variance across rounds: {metrics_df["accuracy"].std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, color in zip(axes, ['accuracy', 'f1', 'roc_auc'], ['steelblue', 'seagreen', 'darkorange']):
    ax.plot(metrics_df['round'], metrics_df[metric], marker='o', color=color, linewidth=2)
    if conv_round and metric == 'accuracy':
        ax.axvline(conv_round, linestyle='--', color='red', alpha=0.7, label=f'Converged @ round {conv_round}')
        ax.legend(fontsize=8)
    ax.set_xlabel('Round')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{metric.replace("_", " ").title()} vs. Round')
    ax.grid(True, alpha=0.3)

plt.suptitle('Federated Learning Convergence Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'convergence_analysis.png', bbox_inches='tight')
plt.show()

## 3. Per-Client Performance

In [ ]:
with open(MODELS_DIR / 'global_model.pkl', 'rb') as f:
    global_model = pickle.load(f)

client_dfs = {i: load_client_data(i) for i in range(N_CLIENTS)}
scaler = get_global_scaler(client_dfs)

client_results = []
for cid, df in client_dfs.items():
    _, _, X_test, y_test, _ = prepare_client_tensors(df, scaler=scaler)
    m = evaluate_model(global_model, X_test, y_test, label=f'Client {cid}')
    m['client_id'] = cid
    m['n_test'] = len(y_test)
    client_results.append(m)

client_perf = pd.DataFrame(client_results).set_index('client_id')
print(client_perf[['accuracy', 'f1', 'roc_auc', 'log_loss', 'n_test']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = sns.color_palette('Set2', N_CLIENTS)

for ax, metric in zip(axes, ['accuracy', 'f1', 'roc_auc']):
    values = [client_perf.loc[i, metric] for i in range(N_CLIENTS)]
    bars = ax.bar([f'Client {i}' for i in range(N_CLIENTS)], values, color=colors)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=9)
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{metric.replace("_", " ").title()} per Client')
    ax.set_ylim(0, 1.1)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Global Model Performance per Client', fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'per_client_performance.png', bbox_inches='tight')
plt.show()

## 4. Communication Cost

In [ ]:
params = get_model_parameters(global_model)
comm_cost = compute_communication_cost(
    n_clients=N_CLIENTS, num_rounds=NUM_ROUNDS, model_params=params
)

print('Communication Cost Analysis')
print(f'  Model parameters:       {comm_cost["n_params"]:,}')
print(f'  Bytes per transmission: {comm_cost["bytes_per_round_per_client"]:,}')
print(f'  Total bytes:            {comm_cost["total_bytes"]:,}')
print(f'  Total MB:               {comm_cost["total_mb"]:.4f} MB')

## 5. Privacy Properties

In [ ]:
print('Privacy Analysis')
print('=' * 50)
print('Raw data privacy:  100% — data never leaves clients')
print('Transmitted:       Model weights only (no raw data)')
print('Model inversion:   Low risk (logistic regression)')
print()

# Estimate privacy budget if DP were applied
eps = estimate_privacy_budget_spent(
    num_rounds=NUM_ROUNDS,
    n_clients=N_CLIENTS,
    delta=1e-5,
    sigma=1.0,
    n_samples=min(len(df) for df in client_dfs.values()),
    batch_size=32
)
print(f'DP budget estimate (sigma=1.0): ε ≈ {eps:.3f}')
print('(Differential privacy not applied in this run)')